In [ ]:
#@title &nbsp;
# The banner is a cell output rather than a text cell, because Colab
# limits text cells to a reading width and outputs use the full window.
from IPython.display import HTML, display
display(HTML('<style>\n.adcp-hdr{box-sizing:border-box;width:100%;display:flex;align-items:center;\n  gap:clamp(14px,2.2vw,34px);padding:clamp(14px,2vw,28px) clamp(16px,2.6vw,40px);\n  border-radius:12px;background:linear-gradient(90deg,#15424e,#0b2e37);overflow:hidden;\n  font-family:system-ui,-apple-system,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif}\n.adcp-mark{flex:0 0 auto;width:clamp(58px,6.4vw,112px);height:clamp(58px,6.4vw,112px);\n  border-radius:22%;background:#134c55;display:flex;align-items:center;justify-content:center}\n.adcp-mark svg{width:80%;height:80%}\n.adcp-txt{flex:1 1 auto;min-width:0}\n.adcp-title{color:#f7f9f9;font-weight:700;font-size:clamp(24px,3.2vw,50px);line-height:1.08}\n.adcp-tag{color:#9fc4c4;font-size:clamp(12.5px,1.25vw,19px);margin-top:.45em}\n.adcp-rule{width:clamp(56px,6vw,110px);height:4px;border-radius:2px;background:#d67a3a;\n  margin:clamp(9px,1vw,15px) 0 clamp(7px,.8vw,12px)}\n.adcp-sub{color:#9fc4c4;font-size:clamp(12px,1.05vw,16px)}\n.adcp-motif{flex:0 1 auto;width:clamp(120px,22vw,420px);align-self:flex-end}\n@media (max-width:760px){.adcp-motif{display:none}}\n.adcp-note{font:13.5px/1.6 system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;margin:12px 2px 2px}\n.adcp-note a{color:#1a8a8a}\n.adcp-small{font-size:12px;opacity:.75}\n</style>\n<div class="adcp-hdr">\n  <div class="adcp-mark"><svg viewBox="0 0 62 62" xmlns="http://www.w3.org/2000/svg" aria-hidden="true"><path d="M46 14 A20 20 0 1 0 46 48" fill="none" stroke="#22a0a0" stroke-width="7" stroke-linecap="round"/><path d="M22 40 L30 35 L38 30 L46 25" fill="none" stroke="#f7f9f9" stroke-width="2.6" stroke-linecap="round"/><circle cx="22" cy="40" r="4.2" fill="#d67a3a"/><circle cx="30" cy="35" r="4.2" fill="#f7f9f9"/><circle cx="38" cy="30" r="4.2" fill="#d67a3a"/><circle cx="46" cy="25" r="4.2" fill="#f7f9f9"/></svg></div>\n  <div class="adcp-txt">\n    <div class="adcp-title">ADCPpilot</div>\n    <div class="adcp-tag">a product of the Scripps Research Institute and the GUI powered by HOPE Lab</div>\n    <div class="adcp-rule"></div>\n    <div class="adcp-sub">Peptide docking, from receptor to ranked table.</div>\n  </div>\n  <svg class="adcp-motif" viewBox="0 0 420 72" preserveAspectRatio="xMaxYMax meet" aria-hidden="true"><polyline points="12,58 78,18 144,58 210,18 276,58 342,18 408,58" fill="none" stroke="#155e65" stroke-width="5" stroke-linejoin="round"/><circle cx="12" cy="58" r="9" fill="#7d5f41"/><circle cx="78" cy="18" r="9" fill="#16686d"/><circle cx="144" cy="58" r="9" fill="#7d5f41"/><circle cx="210" cy="18" r="9" fill="#16686d"/><circle cx="276" cy="58" r="9" fill="#7d5f41"/><circle cx="342" cy="18" r="9" fill="#16686d"/><circle cx="408" cy="58" r="9" fill="#7d5f41"/></svg>\n</div>\n<div class="adcp-note">A graphical interface to <a href="https://ccsb.scripps.edu/adcp/" target="_blank">ADCP</a>, a product of the Scripps Research Institute. Powered by HOPE Lab. Not affiliated with or endorsed by Scripps.<br>\nOne press starts everything: press &#9654; on <b>Start ADCPpilot</b> below. The first press installs the tool, about a quarter of an hour, and then opens the interface here in the notebook. The list of runs includes a finished example, MDM2 with three peptides, to open and explore. The docking runs on this Colab machine, which suits a trial run; for larger searches, install it on a <a href="https://github.com/aadhilhaq/ADCP_docking/blob/main/INSTALL.md#1-on-a-slurm-cluster" target="_blank">Slurm cluster</a>.<br>\n<span class="adcp-small">The first press installs the ADCP 1.1 suite, which contains ADCP, with the Scripps Research Institute&#39;s own recipe, and optionally AmberTools from conda-forge. Parts of the suite are free for non-commercial use only.</span></div>\n'))


In [ ]:
#@title **▶ Start ADCPpilot** &nbsp; <font color="#5f7070">one press. The first one installs the tool, about 15 minutes, then opens the interface below</font>
add_MM_GBSA = True #@param {type:"boolean"}
#@markdown Press ▶ on this step. The first press installs what the session needs, the ADCP 1.1 suite among it, and opens the interface here in the notebook; pressing it again opens the interface again without installing anything. Tick **add_MM_GBSA** to install AmberTools for MM-GBSA rescoring.
import importlib, importlib.util, os, re, subprocess, sys, time
from IPython.display import HTML, Javascript, display
from google.colab import output

PORT = 8765
REPO = "https://github.com/aadhilhaq/ADCP_docking"
RUNS = "/content/adcp_runs"
AMBER = "/content/amber"
ADCP11 = "/content/adcpsuite"
SERVE_LOG = "/content/adcp_pilot_serve.log"

# One status line, updated in place, so the cell ends with a single line.
_status = display(HTML(""), display_id=True)


def _show(text, colour="#5f7070"):
    _status.update(HTML('<div style="font:14px system-ui;color:%s;margin:2px 0">'
                        '%s</div>' % (colour, text)))


# What is already here. Each test looks for the program itself, so a second
# press, or a session that was interrupted part way, installs only what is
# missing.
def _has_interface():
    importlib.invalidate_caches()
    return importlib.util.find_spec("adcp_pilot") is not None


def _has_amber():
    return os.path.isfile(os.path.join(AMBER, "bin", "tleap"))


def _has_adcp11():
    return (os.path.isfile(os.path.join(ADCP11, "bin", "adcp"))
            and os.path.isfile(os.path.join(ADCP11, "bin", "python")))


def _can_fetch_code():
    """Whether this session can fetch the code from GitHub.

    pip installs the interface from a clone of REPO, so this asks git the
    same question first, and with whatever access this session already has:
    a session that can reach the repository is never stopped here. The two
    settings keep git from waiting for a username nobody is there to type.
    """
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS="/bin/true")
    try:
        return subprocess.run(["git", "ls-remote", REPO, "HEAD"], env=env,
                              capture_output=True, text=True,
                              timeout=90).returncode == 0
    except Exception:
        return False


def _steps():
    steps = []
    if not _has_interface():
        steps.append((
            "Installing the interface",
            [sys.executable, "-m", "pip", "-q", "install", "rdkit", "pdb2pqr",
             "openmm", "pdbfixer",
             "adcp-pilot @ git+" + REPO]))
    if not _has_adcp11():
        # The ADCP 1.1 suite, which every run docks with: Scripps' own
        # recipe, as a separate Python 3.7 environment, with each Scripps
        # package checked against a pinned SHA-256. The interface finds it
        # at /content/adcpsuite.
        steps.append((
            "Installing the ADCP 1.1 suite from the Scripps Research Institute",
            [sys.executable, "-m", "adcp_pilot.adcp11_install",
             "--dest", ADCP11]))
    if add_MM_GBSA and not _has_amber():
        # AmberTools is on conda-forge, not pip. micromamba installs it into
        # its own folder, so Colab's Python and the notebook are untouched.
        steps.append((
            "Installing AmberTools for MM-GBSA from conda-forge",
            ["bash", "-c",
             "set -e; mkdir -p /content/micromamba; cd /content/micromamba; "
             "[ -x bin/micromamba ] || curl -Ls "
             "https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba; "
             "./bin/micromamba create -y -q "
             "-r /content/micromamba -p /content/amber -c conda-forge ambertools"]))
    return steps


def _install(steps):
    for i, (label, cmd) in enumerate(steps):
        _show("%s (%d of %d) …" % (label, i + 1, len(steps)))
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode:
            said = "%s did not finish. What it printed is below." % label
            _show("✗ %s Press ▶ again to carry on from here." % said,
                  "#b3412e")
            print(said)
            # Both streams, but not the same text twice, which is how pip
            # reports a step that its own subprocess ended.
            out = p.stdout or ""
            if (p.stderr or "").strip() and (p.stderr or "").strip() not in out:
                out += p.stderr
            print(out[-4000:])
            return False
    return True


def _start():
    """Start the interface and return its process and its one time token."""
    os.makedirs(RUNS, exist_ok=True)
    # A finished example run to open before docking anything: MDM2 with
    # three peptides. Left as it is when it is already there.
    ex = subprocess.run([sys.executable, "-m", "adcp_pilot.cli", "example",
                         "--runs", RUNS], capture_output=True, text=True)
    if ex.returncode:
        print("The example run could not be added: %s"
              % (ex.stdout + ex.stderr).strip()[-300:])
    # ADCP_EXECUTOR=local runs the work on this machine; Colab has no Slurm.
    # The suite is named rather than put on PATH: its bin folder holds its
    # own python, which would then be every program's python.
    env = dict(os.environ, ADCP_EXECUTOR="local", ADCPSUITE_HOME=ADCP11)
    # AmberTools, for MM-GBSA. Appended to PATH, so its own python and tools
    # never shadow Colab's.
    if _has_amber():
        env["AMBERHOME"] = AMBER
        env["PATH"] = env["PATH"] + ":" + os.path.join(AMBER, "bin")
    # The server writes to a file rather than a pipe, so it never stops to
    # wait for the notebook to read what it printed.
    log = open(SERVE_LOG, "w")
    proc = subprocess.Popen(
        [sys.executable, "-u", "-m", "adcp_pilot.cli", "serve",
         "--host", "127.0.0.1", "--port", str(PORT), "--runs", RUNS],
        env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
    token, deadline = None, time.time() + 90
    while token is None and time.time() < deadline:
        with open(SERVE_LOG, errors="replace") as fh:
            said = fh.read()
        m = re.search(r"http://127\.0\.0\.1:\d+/\?t=([A-Za-z0-9_\-]+)", said)
        if m:
            return proc, m.group(1)
        if proc.poll() is not None:
            break
        time.sleep(0.4)
    _show("✗ The interface did not start. Its output is below.", "#b3412e")
    with open(SERVE_LOG, errors="replace") as fh:
        print("".join(fh.read().splitlines(True)[-30:]))
    return proc, None


def _open(token):
    try:
        output.no_vertical_scroll()
    except Exception:
        pass
    output.serve_kernel_port_as_window(
        PORT, path="/?t=" + token, anchor_text="Open in a separate tab ↗")
    # embed=1 leaves the banner above to this notebook and lets the form
    # use the full width of the frame.
    output.serve_kernel_port_as_iframe(
        PORT, path="/?t=%s&embed=1" % token, width="100%", height="800")
    # Size the frame to the visible window, so the interface scrolls
    # inside it like an application window.
    display(Javascript("""
(function fit(tries) {
  var f = document.querySelector('iframe');
  if (!f) { if (tries < 100) setTimeout(function () { fit(tries + 1); }, 150); return; }
  f.style.width = '100%';
  f.style.display = 'block';
  f.style.border = '1px solid #d5e0e0';
  f.style.borderRadius = '10px';
  var size = function () {
    f.style.height = Math.max(620, (window.outerHeight || screen.availHeight) - 230) + 'px';
  };
  size();
  window.addEventListener('resize', size);
})(0);
"""))


_t0 = time.time()
_todo = _steps()
_needs_code = any(label == "Installing the interface" for label, _ in _todo)
if _needs_code:
    _show("Checking that the code can be fetched from GitHub …")
# Nothing is installed when it cannot be, and nothing is printed: a visitor
# can do nothing about a repository that is not public yet, so the clone's
# own complaint would only be noise.
_blocked = _needs_code and not _can_fetch_code()
if _blocked:
    _show("ADCPpilot is not public yet, so this session cannot "
          "fetch it from GitHub and the notebook cannot install it. It will "
          "work here as soon as the repository is released. If it has been "
          "released, press ▶ again: reaching GitHub can fail for a moment.",
          "#b3412e")
elif _todo:
    _show("Setting up this session: %d %s to do …"
          % (len(_todo), "step" if len(_todo) == 1 else "steps"))
if not _blocked and _install(_todo):
    _m, _s = divmod(int(time.time() - _t0), 60)
    _took = ("%d min %02d s" % (_m, _s)) if _m else ("%d s" % _s)
    # A server already running in this session is kept, so a run in progress
    # is not interrupted; it is restarted when something has just been added
    # to the session, which it has to be told about.
    _live = (globals().get("_SERVER") is not None
             and _SERVER.poll() is None and globals().get("_TOKEN"))
    if _live and not _todo:
        _show("Opening the interface again …")
    else:
        if globals().get("_SERVER") is not None and _SERVER.poll() is None:
            _SERVER.terminate()
            _SERVER.wait(10)
        _show("Starting the interface …")
        _SERVER, _TOKEN = _start()
    if _TOKEN:
        if _todo:
            _extra = [x for x, on in (("MM-GBSA", add_MM_GBSA),) if on]
            _head = ("Installed in %s%s"
                     % (_took, (", with " + " and ".join(_extra)) if _extra else ""))
        else:
            _head = "Already set up in this session"
        _show("✓ %s; the interface is below." % _head, "#1a8a8a")
        _open(_TOKEN)


In [ ]:
#@title **Download a run** &nbsp; <font color="#5f7070">optional, once a run has finished: the interface has its own download buttons</font>
RUN = "" #@param {type:"string"}
#@markdown Leave empty for the latest run. The zip includes HOW_TO_CITE.txt, CITATION.bib and CITATION.ris.
import os, shutil
from IPython.display import HTML, display
from google.colab import files

ROOT = "/content/adcp_runs"


def _say(text, colour="#b3412e"):
    display(HTML('<div style="font:14px system-ui;color:%s">%s</div>'
                 % (colour, text)))


runs = sorted((os.path.join(ROOT, d) for d in
               (os.listdir(ROOT) if os.path.isdir(ROOT) else [])
               if os.path.isfile(os.path.join(ROOT, d, "job.json"))),
              key=os.path.getmtime)
# Pressed before the interface was ever started, or before anything was
# docked: say what to press instead of failing.
if not runs:
    _say("No runs here yet. Press ▶ on <b>Start ADCPpilot</b> "
         "above, dock something, then come back to this step.")
elif RUN and not os.path.isfile(os.path.join(ROOT, RUN, "job.json")):
    _say("There is no run named <b>%s</b>. This session has: %s. Leave RUN "
         "empty for the latest one."
         % (RUN, ", ".join(os.path.basename(r) for r in runs)))
else:
    pick = os.path.join(ROOT, RUN) if RUN else runs[-1]
    z = shutil.make_archive("/content/" + os.path.basename(pick), "zip", pick)
    _say("✓ %s (%.1f MB)" % (os.path.basename(z), os.path.getsize(z) / 1e6),
         "#1a8a8a")
    files.download(z)
